# Visual Computer-Use AI Agent
Screenshot understanding, OCR-based UI extraction, natural-language task planning, and human-approved action proposals.

This safe reference notebook observes screenshots and proposes UI actions. It does **not** click, type, submit forms, purchase items, send messages, or control the operating system. Bounding boxes are observations, not verified click targets.

In [ ]:
import re, json, hashlib
from dataclasses import dataclass, asdict
from pathlib import Path

try:
    from PIL import Image
    import matplotlib.pyplot as plt
    IMAGE_TOOLS = True
except Exception:
    IMAGE_TOOLS = False

try:
    import pytesseract
    OCR_AVAILABLE = True
except Exception:
    OCR_AVAILABLE = False

print("Image tools:", IMAGE_TOOLS, "| OCR package:", OCR_AVAILABLE)

## 1. Load a screenshot and extract text (optional)

In [ ]:
SCREENSHOT_PATH = "sample_ui_screenshot.png"

def load_screenshot(path):
    if not IMAGE_TOOLS:
        raise RuntimeError("Install pillow and matplotlib.")
    img = Image.open(path).convert("RGB")
    print("Image size:", img.size)
    plt.figure(figsize=(12,7)); plt.imshow(img); plt.axis("off"); plt.show()
    return img

def extract_ocr(img):
    if not OCR_AVAILABLE:
        raise RuntimeError("Install pytesseract and the Tesseract OCR system engine.")
    data = pytesseract.image_to_data(img, output_type=pytesseract.Output.DICT)
    rows=[]
    for i, txt in enumerate(data["text"]):
        txt=txt.strip()
        if not txt: continue
        try: conf=float(data["conf"][i])
        except Exception: conf=-1
        x,y,w,h=[int(data[k][i]) for k in ("left","top","width","height")]
        rows.append({"text":txt,"confidence":conf,"bbox":[x,y,x+w,y+h]})
    return rows

# Uncomment after adding your own screenshot:
# screenshot = load_screenshot(SCREENSHOT_PATH)
# observations = extract_ocr(screenshot)

## 2. Convert observations into UI elements

In [ ]:
@dataclass
class UIElement:
    element_id: str
    text: str
    kind_hint: str
    bbox: list
    confidence: float

def infer_kind(text):
    t=text.lower()
    if any(x in t for x in ["save","submit","continue","search","cancel","close","sign in"]):
        return "possible_button"
    if t in {"settings","home","profile","help","menu"}:
        return "possible_navigation"
    return "text"

def to_elements(rows):
    return [UIElement(f"ui_{i+1}",r["text"],infer_kind(r["text"]),r["bbox"],r["confidence"])
            for i,r in enumerate(rows)]

# Synthetic demo observations; replace with OCR output when available.
demo_rows=[
 {"text":"Dashboard","confidence":96,"bbox":[20,20,150,55]},
 {"text":"Settings","confidence":95,"bbox":[25,140,130,180]},
 {"text":"Notifications","confidence":93,"bbox":[210,160,370,195]},
 {"text":"Save","confidence":92,"bbox":[820,620,900,660]}
]
elements=to_elements(demo_rows)
for e in elements: print(asdict(e))

## 3. Parse the user's task and rank visible elements

In [ ]:
task = "Open Settings and locate notification preferences. Do not change anything."

def parse_task(text):
    low=text.lower()
    return {
      "text":text,
      "terms":[w for w in re.findall(r"[a-z]+",low) if len(w)>3],
      "no_change":any(x in low for x in ["do not","don't","without changing"]),
      "sensitive":any(x in low for x in ["password","payment","bank","private"])
    }

def rank_elements(parsed, elements):
    q=set(parsed["terms"]); ranked=[]
    for e in elements:
        words=set(re.findall(r"[a-z]+",e.text.lower()))
        score=len(q & words) + (0.2 if e.kind_hint=="possible_navigation" else 0)
        score += max(e.confidence,0)/1000
        ranked.append((e,score))
    return sorted(ranked,key=lambda x:x[1],reverse=True)

parsed=parse_task(task)
ranked=rank_elements(parsed,elements)
for e,s in ranked: print(round(s,3),e.element_id,e.text,e.kind_hint)

## 4. Generate a safe action proposal

In [ ]:
@dataclass
class ProposedAction:
    step: int
    action: str
    target_id: str | None
    target_label: str | None
    reason: str
    risk: str
    requires_confirmation: bool

def create_plan(parsed, ranked):
    low=parsed["text"].lower(); plan=[]
    if "settings" in low:
        match=next((e for e,_ in ranked if "settings" in e.text.lower()),None)
        if match:
            plan.append(ProposedAction(1,"navigate_to_or_select",match.element_id,match.text,
                "Visible label matches the requested destination; verify manually.","low",True))
        else:
            plan.append(ProposedAction(1,"request_clarification",None,None,
                "Settings was not detected; do not guess coordinates.","low",False))
    if "notification" in low:
        plan.append(ProposedAction(len(plan)+1,"locate_section",None,"Notifications",
            "Look for the Notifications section; do not change its settings.","low",False))
    if any(x in low for x in ["delete","purchase","buy","send","submit","pay","transfer"]):
        plan.append(ProposedAction(len(plan)+1,"stop_for_human_review",None,None,
            "Consequential action detected; do not proceed automatically.","high",True))
    return plan or [ProposedAction(1,"request_clarification",None,None,"Task intent is unclear.","low",False)]

plan=create_plan(parsed,ranked)
print(json.dumps([asdict(x) for x in plan],indent=2))

## 5. Human approval gate and audit export

In [ ]:
approval_record={
 "task":task,
 "human_review_required":True,
 "execution_enabled":False,
 "notice":"No UI action is executed by this notebook. Review target and intent manually.",
 "elements":[asdict(e) for e in elements],
 "proposed_actions":[asdict(x) for x in plan]
}
with open("visual_agent_action_proposal.json","w",encoding="utf-8") as f:
    json.dump(approval_record,f,indent=2,ensure_ascii=False)
print("Exported visual_agent_action_proposal.json")
print("Execution enabled:",approval_record["execution_enabled"])

## 6. Production roadmap and limitations

- Add a vision-language model for richer screenshot interpretation.
- Integrate accessibility trees where authorized and available.
- Verify UI targets before proposing actions; never rely on guessed coordinates.
- Add policy checks, user permissions, human confirmation, and audit logs.
- Test only in sandbox applications before any real-world integration.

**Limitations:** OCR does not prove that a label is clickable. The included task planner is rule-based, not an autonomous multimodal LLM. No browser, mouse, keyboard, or operating-system control is implemented. Avoid screenshots containing passwords, payment details, or private information.